In [2]:
pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


#### Import Necessary Dependencies

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

In [2]:
# Initializing Spark Session
spark = SparkSession.builder.appName("NugaBankETL").getOrCreate()

In [3]:
spark

#### Data Extraction

In [4]:
nuga_bank_df = spark.read.csv(r'C:\Users\USER\Desktop\Nuga_bank\dataset\nuga_bank_transactions.csv', header=True, inferSchema=True)

In [5]:
nuga_bank_df.show()

+--------------------+------+----------------+-----------------+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+--------------------+-------------+-------------+--------+-----+---------+--------------------+--------------------+------+--------------+
|    Transaction_Date|Amount|Transaction_Type|    Customer_Name|    Customer_Address|       Customer_City|Customer_State|    Customer_Country|             Company|           Job_Title|               Email|       Phone_Number| Credit_Card_Number|                IBAN|Currency_Code|Random_Number|Category|Group|Is_Active|        Last_Updated|         Description|Gender|Marital_Status|
+--------------------+------+----------------+-----------------+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------

In [6]:
nuga_bank_df.printSchema()

root
 |-- Transaction_Date: timestamp (nullable = true)
 |-- Amount: double (nullable = true)
 |-- Transaction_Type: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Customer_Address: string (nullable = true)
 |-- Customer_City: string (nullable = true)
 |-- Customer_State: string (nullable = true)
 |-- Customer_Country: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone_Number: string (nullable = true)
 |-- Credit_Card_Number: long (nullable = true)
 |-- IBAN: string (nullable = true)
 |-- Currency_Code: string (nullable = true)
 |-- Random_Number: double (nullable = true)
 |-- Category: string (nullable = true)
 |-- Group: string (nullable = true)
 |-- Is_Active: string (nullable = true)
 |-- Last_Updated: timestamp (nullable = true)
 |-- Description: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Marital_Status: string (nullable = true)

### Data Transformation

In [7]:
# View columns
nuga_bank_df.columns

['Transaction_Date',
 'Amount',
 'Transaction_Type',
 'Customer_Name',
 'Customer_Address',
 'Customer_City',
 'Customer_State',
 'Customer_Country',
 'Company',
 'Job_Title',
 'Email',
 'Phone_Number',
 'Credit_Card_Number',
 'IBAN',
 'Currency_Code',
 'Random_Number',
 'Category',
 'Group',
 'Is_Active',
 'Last_Updated',
 'Description',
 'Gender',
 'Marital_Status']

In [8]:
# Check number of rows
num_rows = nuga_bank_df.count()
print(f"Number of rows: {num_rows}")

Number of rows: 1000000


In [10]:
# Check number of columns
num_columns = len(nuga_bank_df.columns)
print(f"Number of columns: {num_columns}")

Number of columns: 23


In [11]:
# Checking for null values
for column in nuga_bank_df.columns:
    print(column, 'Nulls', nuga_bank_df.filter(nuga_bank_df[column].isNull()).count())

Transaction_Date Nulls 0
Amount Nulls 0
Transaction_Type Nulls 0
Customer_Name Nulls 100425
Customer_Address Nulls 100087
Customer_City Nulls 100034
Customer_State Nulls 100009
Customer_Country Nulls 100672
Company Nulls 100295
Job_Title Nulls 99924
Email Nulls 100043
Phone_Number Nulls 100524
Credit_Card_Number Nulls 100085
IBAN Nulls 100300
Currency_Code Nulls 99342
Random_Number Nulls 99913
Category Nulls 100332
Group Nulls 100209
Is_Active Nulls 100259
Last_Updated Nulls 100321
Description Nulls 100403
Gender Nulls 99767
Marital_Status Nulls 99904


In [13]:
# Fill null values with 0, 0.0, or 'Unknown' based on the data type of the column
nuga_bank_df_clean = nuga_bank_df.fillna({
    'Customer_Name': 'Unknown',
    'Customer_Address': 'Unknown',
    'Customer_City': 'Unknown',
    'Customer_State': 'Unknown',
    'Customer_Country': 'Unknown',
    'Company': 'Unknown',
    'Job_Title': 'Unknown',
    'Email': 'Unknown',
    'Phone_Number': 'Unknown',
    'Credit_Card_Number': 0,
    'IBAN': 'Unknown',
    'Currency_Code': 'Unknown',
    'Random_Number': 0.0,
    'Category': 'Unknown',
    'Group': 'Unknown',
    'Is_Active': 'Unknown',
    'Description': 'Unknown',
    'Gender': 'Unknown',
    'Marital_Status': 'Unknown'

})


In [14]:
# Drop row where last_updated is null
nuga_bank_df_clean = nuga_bank_df_clean.na.drop(subset=['Last_Updated'])

In [15]:
 # Checking for null values
for column in nuga_bank_df_clean.columns:
    print(column, 'Nulls', nuga_bank_df_clean.filter(nuga_bank_df_clean[column].isNull()).count())

Transaction_Date Nulls 0
Amount Nulls 0
Transaction_Type Nulls 0
Customer_Name Nulls 0
Customer_Address Nulls 0
Customer_City Nulls 0
Customer_State Nulls 0
Customer_Country Nulls 0
Company Nulls 0
Job_Title Nulls 0
Email Nulls 0
Phone_Number Nulls 0
Credit_Card_Number Nulls 0
IBAN Nulls 0
Currency_Code Nulls 0
Random_Number Nulls 0
Category Nulls 0
Group Nulls 0
Is_Active Nulls 0
Last_Updated Nulls 0
Description Nulls 0
Gender Nulls 0
Marital_Status Nulls 0


In [16]:
# Check new number of rows
num_rows = nuga_bank_df_clean.count()
print(f"Number of rows: {num_rows}")

Number of rows: 899679


In [17]:
# To view the summary statistics of the dataset
nuga_bank_df_clean.describe().show()

+-------+------------------+----------------+-------------+--------------------+-------------+--------------+----------------+-------------+------------------+-------------------+--------------------+--------------------+--------------------+-------------+-----------------+--------+-------+---------+--------------------+-------+--------------+
|summary|            Amount|Transaction_Type|Customer_Name|    Customer_Address|Customer_City|Customer_State|Customer_Country|      Company|         Job_Title|              Email|        Phone_Number|  Credit_Card_Number|                IBAN|Currency_Code|    Random_Number|Category|  Group|Is_Active|         Description| Gender|Marital_Status|
+-------+------------------+----------------+-------------+--------------------+-------------+--------------+----------------+-------------+------------------+-------------------+--------------------+--------------------+--------------------+-------------+-----------------+--------+-------+---------+-------

#### Data Modelling

In [20]:
nuga_bank_df_clean.columns

['Transaction_Date',
 'Amount',
 'Transaction_Type',
 'Customer_Name',
 'Customer_Address',
 'Customer_City',
 'Customer_State',
 'Customer_Country',
 'Company',
 'Job_Title',
 'Email',
 'Phone_Number',
 'Credit_Card_Number',
 'IBAN',
 'Currency_Code',
 'Random_Number',
 'Category',
 'Group',
 'Is_Active',
 'Last_Updated',
 'Description',
 'Gender',
 'Marital_Status']

##### Transaction Table

In [21]:
transaction = nuga_bank_df_clean.select('Transaction_date', 'Amount', 'Transaction_Type')

In [22]:
# Adding transaction_id column
transaction = transaction.withColumn('transaction_id', monotonically_increasing_id())

In [23]:
#Reorder columns
transaction = transaction.select('transaction_id', 'Transaction_date', 'Amount', 'Transaction_Type')

In [24]:
transaction.show()

+--------------+--------------------+------+----------------+
|transaction_id|    Transaction_date|Amount|Transaction_Type|
+--------------+--------------------+------+----------------+
|             0|2024-03-23 15:38:...| 34.76|      Withdrawal|
|             1|2024-04-22 19:15:...|163.92|      Withdrawal|
|             2|2024-04-12 19:46:...|386.32|      Withdrawal|
|             3|2024-04-17 15:29:...|407.15|         Deposit|
|             4|2024-02-10 01:51:...|161.31|         Deposit|
|             5|2024-02-10 22:56:...|764.34|        Transfer|
|             6|2024-04-07 00:07:...|734.59|         Deposit|
|             7|2024-03-08 01:51:...|592.43|         Deposit|
|             8|2024-02-01 12:34:...| 927.1|         Deposit|
|             9|2024-03-22 16:46:...| 66.59|        Transfer|
|            10|2024-04-23 13:30:...| 246.3|      Withdrawal|
|            11|2024-01-13 01:22:...|782.32|      Withdrawal|
|            12|2024-02-25 15:16:...|818.42|      Withdrawal|
|       

##### Customer Table

In [25]:
customer = nuga_bank_df_clean.select('Customer_Name', 'Customer_Address', 'Customer_City', 'Customer_State', 'Customer_Country', 'Email', 'Phone_Number').distinct()   

# Adding customer_id column
customer = customer.withColumn('customer_id', monotonically_increasing_id())

# Reorder columns
customer = customer.select('customer_id', 'Customer_Name', 'Customer_Address', 'Customer_City', 'Customer_State', 'Customer_Country', 'Email', 'Phone_Number')  


In [26]:
customer.count()

899679

In [27]:
customer.show()

+-----------+------------------+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+
|customer_id|     Customer_Name|    Customer_Address|       Customer_City|Customer_State|    Customer_Country|               Email|        Phone_Number|
+-----------+------------------+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+
|          0|    Miguel Leonard|262 Beck Expressw...|             Unknown| West Virginia|             Eritrea| zweaver@example.net|             Unknown|
|          1|           Unknown|             Unknown|         Evanchester|        Oregon|             Uruguay|             Unknown| (384)778-9942x91236|
|          2|    Michael Murphy|894 Williams Ridg...|       Dominguezview|      New York|              Sweden|kristinstanley@ex...|+1-693-739-2204x8851|
|          3|    Tina Gutierrez|    415 Taylor Knoll|           Donnastad|South Ca

##### Employee Table

In [29]:
employee = nuga_bank_df_clean.select('Company', 'Job_Title', 'Gender', 'Marital_Status').distinct()

# Adding employee_id column
employee = employee.withColumn('employee_id', monotonically_increasing_id())

# Reorder columns
employee = employee.select('employee_id', 'Company', 'Job_Title', 'Gender', 'Marital_Status')


In [31]:
employee.count()

809052

In [32]:
employee.show()

+-----------+--------------------+--------------------+-------+--------------+
|employee_id|             Company|           Job_Title| Gender|Marital_Status|
+-----------+--------------------+--------------------+-------+--------------+
|          0|         Price Group|             Unknown|   Male|        Single|
|          1|Rhodes, King and ...| Trade mark attorney|   Male|       Unknown|
|          2|Schmidt, Morgan a...|     Engineer, water| Female|        Single|
|          3|       Johnson Group|  Forensic scientist|   Male|       Unknown|
|          4|     Phillips-Prince|Production assist...|Unknown|        Single|
|          5|      Henry and Sons|Engineer, civil (...| Female|       Married|
|          6|Thompson, Johnson...|Exercise physiolo...|  Other|       Unknown|
|          7|Hernandez, Johnso...|Forensic psycholo...|Unknown|      Divorced|
|          8|Carrillo, Schwart...| Solicitor, Scotland| Female|        Single|
|          9|         Olson-Lucas| Magazine journali

##### Fact Table

In [33]:
fact_table = nuga_bank_df_clean.join(customer,['Customer_Name', 'Customer_Address', 'Customer_City', 'Customer_State', 'Customer_Country', 'Email', 'Phone_Number'],'left')\
.join(transaction, ['Transaction_date', 'Amount', 'Transaction_Type'], 'left')\
.join(employee, ['Company', 'Job_Title', 'Gender', 'Marital_Status'], 'left')\
.select('customer_id', 'transaction_id', 'employee_id', 'Last_Updated', 'Credit_Card_Number', 'IBAN', 'Currency_Code', 'Random_Number', 'Category', 'Group', 'Is_Active', 'Description')    



In [34]:
fact_table.show()

+-----------+--------------+-----------+--------------------+------------------+--------------------+-------------+-------------+--------+-------+---------+--------------------+
|customer_id|transaction_id|employee_id|        Last_Updated|Credit_Card_Number|                IBAN|Currency_Code|Random_Number|Category|  Group|Is_Active|         Description|
+-----------+--------------+-----------+--------------------+------------------+--------------------+-------------+-------------+--------+-------+---------+--------------------+
| 8590097714|   17179869198|     143861|2023-10-12 22:25:...|   180067592769732|GB92JVMY004197871...|          EGP|       7198.0|       A|      Z|      Yes|Before story prof...|
|25769812854|            18|      41531|2020-01-19 18:19:...|   213112163828334|GB50TJFN039979307...|          SVC|       7382.0|       B|      Z|      Yes|Great evening so ...|
|25769976749|   25769803779|     196153|2021-12-07 15:35:...|                 0|GB32LGFL895760023...|         

##### Output the transformed data to parquet

In [ ]:
# Transaction data
transaction.write.mode('overwrite').parquet(r'dataset/transaction')
# Customer data
customer.write.mode('overwrite').parquet(r'dataset/customer')
# Employee data
employee.write.mode('overwrite').parquet(r'dataset/employee')
# Fact table
fact_table.write.mode('overwrite').parquet(r'dataset/fact_table')

##### Output the transform data as csv

In [ ]:
# Transaction data
transaction.repartition(3).write.mode('overwrite').option('header', 'true').csv(r'dataset/transformeddata/csv/transaction')
# Customer data
customer.repartition(3).write.mode('overwrite').option('header', 'true').csv(r'dataset/transformeddata/csv/customer')
# Employee data
employee.repartition(3).write.mode('overwrite').option('header', 'true').csv(r'dataset/transformeddata/csv/employee')
# Fact table
fact_table.repartition(3).write.mode('overwrite').option('header', 'true').csv(r'dataset/transformeddata/csv/fact_table')

### Load Data

In [35]:
# Convert Spark DataFrame to Pandas DataFrame
transaction_pd = transaction.toPandas()
customer_pd = customer.toPandas()
employee_pd = employee.toPandas()
fact_table_pd = fact_table.toPandas()

In [ ]:
# Loading data into PostgreSQL DB
load_dotenv()
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')


In [51]:
# Create connection url
db_url = f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'

# Create SQLAlchemy engine
engine = create_engine(db_url)

# Connect to postgreSQL server
with engine.connect() as connection:
    # Load transaction data
    transaction_pd.to_sql('transaction', connection, if_exists='replace', index=False)
    # Load customer data
    customer_pd.to_sql('customer', connection, if_exists='replace', index=False)
    # Load employee data
    employee_pd.to_sql('employee', connection, if_exists='replace', index=False)
    # Load fact table data
    fact_table_pd.to_sql('fact_table', connection, if_exists='replace', index=False)

print('Data loaded successfully into PostgreSQL database.')

Data loaded successfully into PostgreSQL database.
